# Notebook 07: Fairness and Subgroup Analysis

## Purpose

Estimate target- and threshold-specific subgroup performance, reference-group gaps, cross-target gap differences, and paired bootstrap uncertainty for sex, race/ethnicity, income, and insurance history.

## Inputs

- Labelled complete-case data and fixed folds
- Thresholded out-of-fold predictions from Notebook 06
- Performance metadata from Notebook 06

## Outputs

- Absolute subgroup metrics and reference-group gaps
- Cross-target gap-difference and bootstrap tables
- Primary subgroup figures and appendix gap figures
- `data/processed/fairness_and_subgroup_metadata.json`

## Dependencies

Run Notebooks 01--06 first. Notebooks 08--09 use the fairness output tables and metadata.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## Conceptual cautions

### Target-dependent error terminology

Every error metric is defined relative to a particular operational target. For example:

- false-negative rate relative to prior reported clinician diagnosis;
- false-negative rate relative to current HbA1c at least 6.5%.

The notebook never treats either target as an unquestionable clinical gold standard.

### Two roles of race/ethnicity

Race/ethnicity has two conceptually separate roles:

1. it is supplied to the primary models as a predictor;
2. it is used as an evaluation subgroup.

The subgroup analysis therefore describes how the **race-aware primary models** behave across race/ethnicity groups. It does not answer whether race/ethnicity should be included in a deployed model. The with-race versus without-race comparison belongs to Notebook 08.

### Reference-group gaps

Reference groups are selected for transparent descriptive comparison:

- sex: Female;
- race/ethnicity: Non-Hispanic White;
- income-to-poverty ratio: 4 or higher;
- insurance history: Continuously insured.

A reference group is not declared clinically ideal or normatively superior. A signed gap is simply

$$
\text{subgroup metric}
-
\text{reference-group metric}.
$$

## Bootstrap design

The bootstrap resamples participants and reuses the completed out-of-fold probabilities, fold-specific thresholds, and held-out classifications from Notebook 06.

The same sampled participant indices are used across:

- both model classes;
- both targets;
- all operating points;
- all subgroup definitions.

This preserves paired comparisons.

The bootstrap does not refit the predictive models or reselect thresholds. Its intervals quantify empirical uncertainty in the subgroup metrics **conditional on the completed out-of-fold prediction procedure**.

## 1. Setup

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import os
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from scipy.special import expit

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 220)


## 2. Fixed configuration

In [ ]:
RANDOM_STATE = 26

N_BOOTSTRAP = int(
    os.environ.get(
        "FAIRNESS_N_BOOTSTRAP",
        "1000",
    )
)

BOOTSTRAP_PROGRESS_EVERY = max(
    1,
    min(50, N_BOOTSTRAP),
)

CONFIDENCE_LEVEL = 0.95
CI_LOWER_QUANTILE = (
    1 - CONFIDENCE_LEVEL
) / 2
CI_UPPER_QUANTILE = (
    1 + CONFIDENCE_LEVEL
) / 2

TARGET_SENSITIVITY_LEVELS = [
    0.70,
    0.80,
    0.90,
]

PRIMARY_TARGET_SENSITIVITY = 0.80

MIN_CALIBRATION_N = 50
MIN_CALIBRATION_POSITIVE_N = 20
MIN_CALIBRATION_NEGATIVE_N = 20

VERY_SMALL_OUTCOME_COUNT = 20
SMALL_OUTCOME_COUNT = 50

MODEL_NAMES = [
    "logistic",
    "ebm",
]

MODEL_DISPLAY_NAMES = {
    "logistic": "Logistic regression",
    "ebm": "Explainable Boosting Machine",
}

TARGET_COLUMNS = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]

TARGET_DISPLAY_NAMES = {
    "self_reported_prior_diagnosis": (
        "Prior reported clinician diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "Current HbA1c at least 6.5%"
    ),
}

TARGET_SHORT_NAMES = {
    "self_reported_prior_diagnosis": (
        "Prior diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "HbA1c ≥ 6.5%"
    ),
}

THRESHOLD_METRICS = [
    "false_negative_rate",
    "false_positive_rate",
    "sensitivity",
    "specificity",
    "positive_prediction_rate",
]

PROBABILITY_METRICS = [
    "brier_score",
    "calibration_intercept",
    "calibration_slope",
]

ALL_METRICS = (
    THRESHOLD_METRICS
    + PROBABILITY_METRICS
)

METRIC_DISPLAY_NAMES = {
    "false_negative_rate": (
        "False-negative rate"
    ),
    "false_positive_rate": (
        "False-positive rate"
    ),
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "positive_prediction_rate": (
        "Positive prediction rate"
    ),
    "brier_score": "Brier score",
    "calibration_intercept": (
        "Calibration intercept"
    ),
    "calibration_slope": (
        "Calibration slope"
    ),
}

METRIC_DEPENDS_ON_THRESHOLD = {
    metric: metric in THRESHOLD_METRICS
    for metric in ALL_METRICS
}

INCOME_GROUP_LEVELS = [
    "Below 1",
    "1 to below 2",
    "2 to below 4",
    "4 or higher",
]

SEX_LEVELS = [
    "Female",
    "Male",
]

RACE_ETHNICITY_LEVELS = [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other or multiracial",
]

INSURANCE_HISTORY_LEVELS = [
    "Continuously insured",
    "Currently insured, past-year gap",
    "Currently uninsured",
]

SUBGROUP_LEVELS = {
    "sex": SEX_LEVELS,
    "race_ethnicity": (
        RACE_ETHNICITY_LEVELS
    ),
    "income_poverty_group": (
        INCOME_GROUP_LEVELS
    ),
    "insurance_history": (
        INSURANCE_HISTORY_LEVELS
    ),
}

SUBGROUP_DISPLAY_NAMES = {
    "sex": "Sex",
    "race_ethnicity": "Race/ethnicity",
    "income_poverty_group": (
        "Income-to-poverty group"
    ),
    "insurance_history": (
        "Insurance history"
    ),
}

REFERENCE_GROUPS = {
    "sex": "Female",
    "race_ethnicity": (
        "Non-Hispanic White"
    ),
    "income_poverty_group": (
        "4 or higher"
    ),
    "insurance_history": (
        "Continuously insured"
    ),
}

print("Bootstrap replicates:", N_BOOTSTRAP)
print(
    "Primary operating point:",
    f"{PRIMARY_TARGET_SENSITIVITY:.0%} training sensitivity",
)


## 3. Project paths

In [ ]:
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = (
    PROJECT_DIR / "data" / "processed"
)
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LABELLED_DATA_PATH = (
    PROCESSED_DIR
    / "nhanes_diabetes_complete_case_labeled.csv"
)

FOLD_ASSIGNMENT_PATH = (
    PROCESSED_DIR
    / "primary_cv_fold_assignments.csv"
)

THRESHOLDED_LONG_PATH = (
    PROCESSED_DIR
    / "thresholded_oof_predictions_long.csv"
)

PRIMARY_THRESHOLD_PATH = (
    PROCESSED_DIR
    / "primary_80_sensitivity_thresholded_oof_predictions.csv"
)

PERFORMANCE_METADATA_PATH = (
    PROCESSED_DIR
    / "performance_and_thresholds_metadata.json"
)

FAIRNESS_METADATA_PATH = (
    PROCESSED_DIR
    / "fairness_and_subgroup_metadata.json"
)

required_paths = [
    LABELLED_DATA_PATH,
    FOLD_ASSIGNMENT_PATH,
    THRESHOLDED_LONG_PATH,
    PRIMARY_THRESHOLD_PATH,
    PERFORMANCE_METADATA_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_text = "\n".join(
        f"- {path}"
        for path in missing_paths
    )

    raise FileNotFoundError(
        "Notebook 07 requires completed outputs from Notebook 06. "
        "The following files are missing:\n"
        f"{missing_text}"
    )

print("Project directory:", PROJECT_DIR)
print("Thresholded predictions:", THRESHOLDED_LONG_PATH)


## 4. Load data and Notebook 06 outputs

In [ ]:
data = pd.read_csv(
    LABELLED_DATA_PATH
)

fold_assignments = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

thresholded_long = pd.read_csv(
    THRESHOLDED_LONG_PATH
)

primary_threshold_file = pd.read_csv(
    PRIMARY_THRESHOLD_PATH
)

with PERFORMANCE_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    performance_metadata = json.load(file)

print("Analytic data:", data.shape)
print(
    "All thresholded predictions:",
    thresholded_long.shape,
)
print(
    "Primary-threshold predictions:",
    primary_threshold_file.shape,
)


## 5. Validate sample, folds, predictions, and operating points

In [ ]:
def sorted_by_id(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    result = dataframe.copy()

    result["id"] = pd.to_numeric(
        result["id"],
        errors="raise",
    ).astype("Int64")

    return (
        result
        .sort_values("id")
        .reset_index(drop=True)
    )


data = sorted_by_id(data)
fold_assignments = sorted_by_id(
    fold_assignments
)

required_data_columns = {
    "id",
    "age",
    "sex",
    "race_ethnicity",
    "income_poverty_ratio",
    "insurance_history",
    "joint_label_code",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    *TARGET_COLUMNS,
}

missing_data_columns = (
    required_data_columns
    .difference(data.columns)
)

if missing_data_columns:
    raise KeyError(
        "The analytic dataset is missing columns: "
        f"{sorted(missing_data_columns)}"
    )

required_prediction_columns = {
    "id",
    "cv_fold",
    "joint_label_code",
    "model",
    "target",
    "observed_outcome",
    "probability",
    "target_training_sensitivity",
    "selected_threshold",
    "predicted_class",
}

missing_prediction_columns = (
    required_prediction_columns
    .difference(
        thresholded_long.columns
    )
)

if missing_prediction_columns:
    raise KeyError(
        "The thresholded prediction file is missing columns: "
        f"{sorted(missing_prediction_columns)}"
    )

for target in TARGET_COLUMNS:
    data[target] = pd.to_numeric(
        data[target],
        errors="raise",
    ).astype(int)

data[
    "confirmed_current_pregnancy"
] = pd.to_numeric(
    data[
        "confirmed_current_pregnancy"
    ],
    errors="raise",
).astype(int)

data[
    "primary_sample_eligible"
] = pd.to_numeric(
    data[
        "primary_sample_eligible"
    ],
    errors="raise",
).astype(int)

data[
    "income_poverty_ratio"
] = pd.to_numeric(
    data["income_poverty_ratio"],
    errors="raise",
).astype(float)

for variable in [
    "sex",
    "race_ethnicity",
    "insurance_history",
    "joint_label_code",
]:
    data[variable] = (
        data[variable]
        .astype("string")
    )

thresholded_long["id"] = (
    pd.to_numeric(
        thresholded_long["id"],
        errors="raise",
    ).astype("Int64")
)

thresholded_long["cv_fold"] = (
    pd.to_numeric(
        thresholded_long["cv_fold"],
        errors="raise",
    ).astype(int)
)

thresholded_long[
    "observed_outcome"
] = pd.to_numeric(
    thresholded_long[
        "observed_outcome"
    ],
    errors="raise",
).astype(int)

thresholded_long[
    "probability"
] = pd.to_numeric(
    thresholded_long["probability"],
    errors="raise",
).astype(float)

thresholded_long[
    "selected_threshold"
] = pd.to_numeric(
    thresholded_long[
        "selected_threshold"
    ],
    errors="raise",
).astype(float)

thresholded_long[
    "predicted_class"
] = pd.to_numeric(
    thresholded_long[
        "predicted_class"
    ],
    errors="raise",
).astype(int)

thresholded_long[
    "target_training_sensitivity"
] = pd.to_numeric(
    thresholded_long[
        "target_training_sensitivity"
    ],
    errors="raise",
).astype(float)

for variable in [
    "joint_label_code",
    "model",
    "target",
]:
    thresholded_long[variable] = (
        thresholded_long[variable]
        .astype("string")
    )

if not data["id"].is_unique:
    raise ValueError(
        "The analytic data contain duplicate participant IDs."
    )

if data[
    "confirmed_current_pregnancy"
].ne(0).any():
    raise ValueError(
        "The analytic data contain a confirmed current pregnancy."
    )

if not data[
    "primary_sample_eligible"
].eq(1).all():
    raise ValueError(
        "The analytic data contain an ineligible participant."
    )

base_ids = set(
    data["id"].astype(int)
)

prediction_ids = set(
    thresholded_long[
        "id"
    ].astype(int)
)

if prediction_ids != base_ids:
    raise ValueError(
        "The thresholded predictions belong to a different analytic sample."
    )

observed_models = set(
    thresholded_long[
        "model"
    ].astype(str)
)

if observed_models != set(
    MODEL_NAMES
):
    raise ValueError(
        "The thresholded predictions do not contain exactly the "
        "logistic and EBM models."
    )

observed_targets = set(
    thresholded_long[
        "target"
    ].astype(str)
)

if observed_targets != set(
    TARGET_COLUMNS
):
    raise ValueError(
        "The thresholded predictions do not contain both primary targets."
    )

observed_sensitivity_levels = sorted(
    thresholded_long[
        "target_training_sensitivity"
    ].unique()
)

if not np.allclose(
    observed_sensitivity_levels,
    TARGET_SENSITIVITY_LEVELS,
):
    raise ValueError(
        "The saved operating points are not exactly 70%, 80%, and 90%."
    )

if not thresholded_long[
    "probability"
].between(0, 1).all():
    raise ValueError(
        "At least one predicted probability lies outside [0, 1]."
    )

if not thresholded_long[
    "selected_threshold"
].between(0, 1).all():
    raise ValueError(
        "At least one selected threshold lies outside [0, 1]."
    )

if not set(
    thresholded_long[
        "predicted_class"
    ].unique()
).issubset({0, 1}):
    raise ValueError(
        "Predicted classes contain values other than 0 and 1."
    )

expected_rows_per_id = (
    len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
    * len(TARGET_SENSITIVITY_LEVELS)
)

rows_per_id = (
    thresholded_long
    .groupby("id")
    .size()
)

if not rows_per_id.eq(
    expected_rows_per_id
).all():
    raise ValueError(
        "At least one participant lacks a complete set of model-target-threshold predictions."
    )

key_duplicate_count = int(
    thresholded_long
    .duplicated(
        subset=[
            "id",
            "model",
            "target",
            "target_training_sensitivity",
        ]
    )
    .sum()
)

if key_duplicate_count != 0:
    raise ValueError(
        "The long prediction file contains duplicate participant-model-target-threshold rows."
    )

comparison = (
    thresholded_long
    .merge(
        data[
            [
                "id",
                "joint_label_code",
                *TARGET_COLUMNS,
            ]
        ],
        on="id",
        how="left",
        validate="many_to_one",
        suffixes=(
            "_prediction",
            "_data",
        ),
    )
)

if not comparison[
    "joint_label_code_prediction"
].astype(str).eq(
    comparison[
        "joint_label_code_data"
    ].astype(str)
).all():
    raise ValueError(
        "The prediction file and analytic data disagree on joint labels."
    )

for target in TARGET_COLUMNS:
    target_rows = (
        comparison["target"].astype(str)
        == target
    )

    if not comparison.loc[
        target_rows,
        "observed_outcome",
    ].eq(
        comparison.loc[
            target_rows,
            target,
        ]
    ).all():
        raise ValueError(
            f"The prediction file and analytic data disagree on {target}."
        )

analysis_id_hash = hashlib.sha256(
    ",".join(
        data["id"]
        .astype(str)
        .tolist()
    ).encode("utf-8")
).hexdigest()

if (
    performance_metadata[
        "analysis_id_sha256"
    ]
    != analysis_id_hash
):
    raise ValueError(
        "Notebook 06 metadata belong to a different participant sample."
    )

if (
    performance_metadata[
        "random_state"
    ]
    != RANDOM_STATE
):
    raise ValueError(
        "Notebook 06 used a different random seed."
    )

if not performance_metadata[
    "threshold_selection"
][
    "held_out_outcomes_used_for_selection"
] is False:
    raise ValueError(
        "Notebook 06 metadata do not confirm leakage-free threshold selection."
    )

print("Cross-notebook validation passed.")


## 6. Construct the fixed income-to-poverty groups

In [ ]:
data[
    "income_poverty_group"
] = pd.cut(
    data[
        "income_poverty_ratio"
    ],
    bins=[
        -np.inf,
        1.0,
        2.0,
        4.0,
        np.inf,
    ],
    labels=(
        INCOME_GROUP_LEVELS
    ),
    right=False,
    ordered=True,
)

if data[
    "income_poverty_group"
].isna().any():
    raise ValueError(
        "At least one participant was not assigned to a fixed income group."
    )

data[
    "income_poverty_group"
] = (
    data[
        "income_poverty_group"
    ].astype("string")
)

for variable, levels in SUBGROUP_LEVELS.items():
    observed_levels = set(
        data[variable]
        .dropna()
        .astype(str)
        .unique()
    )

    unexpected_levels = (
        observed_levels
        .difference(levels)
    )

    if unexpected_levels:
        raise ValueError(
            f"{variable} contains unexpected levels: "
            f"{sorted(unexpected_levels)}"
        )

    absent_levels = (
        set(levels)
        .difference(observed_levels)
    )

    if absent_levels:
        raise ValueError(
            f"{variable} is missing prespecified levels: "
            f"{sorted(absent_levels)}"
        )

    reference = (
        REFERENCE_GROUPS[
            variable
        ]
    )

    if reference not in observed_levels:
        raise ValueError(
            f"The reference group {reference!r} is absent from {variable}."
        )

subgroup_counts = []

for variable, levels in SUBGROUP_LEVELS.items():
    for level in levels:
        subgroup_counts.append(
            {
                "subgroup_variable": variable,
                "subgroup_variable_display_name": (
                    SUBGROUP_DISPLAY_NAMES[
                        variable
                    ]
                ),
                "subgroup": level,
                "reference_group": (
                    REFERENCE_GROUPS[
                        variable
                    ]
                ),
                "is_reference_group": bool(
                    level
                    == REFERENCE_GROUPS[
                        variable
                    ]
                ),
                "n": int(
                    data[variable]
                    .eq(level)
                    .sum()
                ),
                "share": float(
                    data[variable]
                    .eq(level)
                    .mean()
                ),
            }
        )

subgroup_counts = pd.DataFrame(
    subgroup_counts
)

subgroup_counts.to_csv(
    TABLE_DIR
    / "fairness_subgroup_counts.csv",
    index=False,
)

subgroup_counts


## 7. Build one participant-level analysis table

The probability vector is identical across the three threshold rows for a given participant, model, and target. The notebook verifies this before constructing a compact participant-level representation.

In [ ]:
probability_consistency = (
    thresholded_long
    .groupby(
        [
            "id",
            "model",
            "target",
        ],
        observed=True,
    )["probability"]
    .agg(
        minimum="min",
        maximum="max",
        unique_n="nunique",
    )
)

if not (
    probability_consistency[
        "unique_n"
    ].eq(1)
    & np.isclose(
        probability_consistency[
            "minimum"
        ],
        probability_consistency[
            "maximum"
        ],
    )
).all():
    raise ValueError(
        "A participant-model-target probability changes across threshold rows."
    )

probability_wide = (
    thresholded_long
    .drop_duplicates(
        subset=[
            "id",
            "model",
            "target",
        ]
    )
    .pivot(
        index="id",
        columns=[
            "model",
            "target",
        ],
        values="probability",
    )
)

probability_wide.columns = [
    (
        "probability__"
        + str(model_name)
        + "__"
        + str(target)
    )
    for model_name, target
    in probability_wide.columns
]

prediction_wide = (
    thresholded_long
    .assign(
        threshold_key=(
            thresholded_long[
                "model"
            ].astype(str)
            + "__"
            + thresholded_long[
                "target"
            ].astype(str)
            + "__sens_"
            + (
                100
                * thresholded_long[
                    "target_training_sensitivity"
                ]
            )
            .round()
            .astype(int)
            .astype(str)
        )
    )
    .pivot(
        index="id",
        columns="threshold_key",
        values="predicted_class",
    )
)

prediction_wide.columns = [
    (
        "predicted_class__"
        + str(column)
    )
    for column
    in prediction_wide.columns
]

participant_analysis = (
    data
    .merge(
        probability_wide
        .reset_index(),
        on="id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        prediction_wide
        .reset_index(),
        on="id",
        how="left",
        validate="one_to_one",
    )
)

expected_probability_columns = [
    (
        "probability__"
        + model_name
        + "__"
        + target
    )
    for model_name in MODEL_NAMES
    for target in TARGET_COLUMNS
]

expected_prediction_columns = [
    (
        "predicted_class__"
        + model_name
        + "__"
        + target
        + "__sens_"
        + str(
            int(
                round(
                    100
                    * target_sensitivity
                )
            )
        )
    )
    for model_name in MODEL_NAMES
    for target in TARGET_COLUMNS
    for target_sensitivity
    in TARGET_SENSITIVITY_LEVELS
]

expected_analysis_columns = (
    expected_probability_columns
    + expected_prediction_columns
)

if participant_analysis[
    expected_analysis_columns
].isna().sum().sum() != 0:
    raise ValueError(
        "The participant-level analysis table contains missing predictions."
    )

participant_analysis = (
    participant_analysis
    .sort_values("id")
    .reset_index(drop=True)
)

print(
    "Participant-level analysis table:",
    participant_analysis.shape,
)


## Section 11 — Primary fairness and subgroup analysis

## 8. Metric and stability helpers

In [ ]:
def safe_ratio(
    numerator: float,
    denominator: float,
) -> float:
    if denominator == 0:
        return np.nan

    return float(
        numerator / denominator
    )


def subgroup_stability_warning(
    n: int,
    positive_n: int,
    negative_n: int,
) -> str:
    minimum_outcome_count = min(
        positive_n,
        negative_n,
    )

    if minimum_outcome_count < VERY_SMALL_OUTCOME_COUNT:
        return (
            "Very unstable: fewer than 20 target-positive "
            "or target-negative participants"
        )

    if minimum_outcome_count < SMALL_OUTCOME_COUNT:
        return (
            "Caution: 20–49 target-positive or "
            "target-negative participants"
        )

    if n < 100:
        return (
            "Caution: subgroup contains fewer than 100 participants"
        )

    return (
        "No prespecified small-cell warning"
    )


def calibration_is_eligible(
    n: int,
    positive_n: int,
    negative_n: int,
    probability: np.ndarray,
) -> bool:
    probability = np.asarray(
        probability,
        dtype=float,
    )

    return bool(
        n >= MIN_CALIBRATION_N
        and positive_n
        >= MIN_CALIBRATION_POSITIVE_N
        and negative_n
        >= MIN_CALIBRATION_NEGATIVE_N
        and np.unique(
            probability
        ).size > 1
    )


def stable_logit(
    probability: np.ndarray,
) -> np.ndarray:
    clipped = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        clipped
        / (
            1 - clipped
        )
    )


def fit_calibration_intercept_slope(
    y_true: np.ndarray,
    probability: np.ndarray,
    maximum_iterations: int = 50,
    tolerance: float = 1e-8,
) -> tuple[float, float, bool]:
    y = np.asarray(
        y_true,
        dtype=float,
    )

    p = np.asarray(
        probability,
        dtype=float,
    )

    n = int(
        y.size
    )

    positive_n = int(
        y.sum()
    )

    negative_n = int(
        n - positive_n
    )

    if not calibration_is_eligible(
        n=n,
        positive_n=positive_n,
        negative_n=negative_n,
        probability=p,
    ):
        return (
            np.nan,
            np.nan,
            False,
        )

    z = stable_logit(
        p
    )

    parameters = np.array(
        [
            0.0,
            1.0,
        ],
        dtype=float,
    )

    design = np.column_stack(
        [
            np.ones(
                n,
                dtype=float,
            ),
            z,
        ]
    )

    converged = False

    for _ in range(
        maximum_iterations
    ):
        linear_predictor = (
            design
            @ parameters
        )

        fitted_probability = expit(
            linear_predictor
        )

        weights = (
            fitted_probability
            * (
                1
                - fitted_probability
            )
        )

        score = (
            design.T
            @ (
                y
                - fitted_probability
            )
        )

        information = (
            design.T
            @ (
                weights[:, None]
                * design
            )
        )

        information = (
            information
            + 1e-10
            * np.eye(2)
        )

        try:
            step = np.linalg.solve(
                information,
                score,
            )
        except np.linalg.LinAlgError:
            return (
                np.nan,
                np.nan,
                False,
            )

        if not np.isfinite(
            step
        ).all():
            return (
                np.nan,
                np.nan,
                False,
            )

        parameters = (
            parameters
            + step
        )

        if np.max(
            np.abs(
                step
            )
        ) < tolerance:
            converged = True
            break

    if not converged:
        return (
            np.nan,
            np.nan,
            False,
        )

    return (
        float(
            parameters[0]
        ),
        float(
            parameters[1]
        ),
        True,
    )


def calculate_probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:
    y = np.asarray(
        y_true,
        dtype=int,
    )

    p = np.asarray(
        probability,
        dtype=float,
    )

    n = int(
        y.size
    )

    positive_n = int(
        y.sum()
    )

    negative_n = int(
        n - positive_n
    )

    (
        calibration_intercept,
        calibration_slope,
        calibration_success,
    ) = fit_calibration_intercept_slope(
        y,
        p,
    )

    return {
        "brier_score": float(
            np.mean(
                (
                    y
                    - p
                ) ** 2
            )
        ),
        "calibration_intercept": (
            calibration_intercept
        ),
        "calibration_slope": (
            calibration_slope
        ),
        "calibration_fit_success": bool(
            calibration_success
        ),
    }


def calculate_threshold_metrics(
    y_true: np.ndarray,
    predicted_class: np.ndarray,
) -> dict:
    y = np.asarray(
        y_true,
        dtype=int,
    )

    predicted = np.asarray(
        predicted_class,
        dtype=int,
    )

    true_positive = int(
        np.sum(
            (y == 1)
            & (predicted == 1)
        )
    )

    false_negative = int(
        np.sum(
            (y == 1)
            & (predicted == 0)
        )
    )

    true_negative = int(
        np.sum(
            (y == 0)
            & (predicted == 0)
        )
    )

    false_positive = int(
        np.sum(
            (y == 0)
            & (predicted == 1)
        )
    )

    sensitivity = safe_ratio(
        true_positive,
        true_positive
        + false_negative,
    )

    specificity = safe_ratio(
        true_negative,
        true_negative
        + false_positive,
    )

    return {
        "true_positive": true_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "false_positive": false_positive,
        "false_negative_rate": (
            1 - sensitivity
            if not pd.isna(
                sensitivity
            )
            else np.nan
        ),
        "false_positive_rate": (
            1 - specificity
            if not pd.isna(
                specificity
            )
            else np.nan
        ),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "positive_prediction_rate": float(
            predicted.mean()
        ),
    }


def target_specific_metric_label(
    metric: str,
    target: str,
) -> str:
    metric_name = (
        METRIC_DISPLAY_NAMES[
            metric
        ]
    )

    target_name = (
        TARGET_DISPLAY_NAMES[
            target
        ]
    )

    return (
        f"{metric_name} relative to {target_name}"
    )


## 9. Calculate absolute subgroup metrics

Brier score and calibration use the out-of-fold probabilities and therefore do not change across thresholds. They are repeated in the combined threshold-specific table for convenience and are marked as threshold-independent.

In [ ]:
absolute_point_rows = []

for (
    subgroup_variable,
    subgroup_levels,
) in SUBGROUP_LEVELS.items():
    for subgroup in subgroup_levels:
        subgroup_mask = (
            participant_analysis[
                subgroup_variable
            ].astype(str)
            == subgroup
        )

        subgroup_data = (
            participant_analysis
            .loc[
                subgroup_mask
            ]
        )

        subgroup_n = int(
            len(
                subgroup_data
            )
        )

        for target in TARGET_COLUMNS:
            y = (
                subgroup_data[
                    target
                ].to_numpy(
                    dtype=int
                )
            )

            positive_n = int(
                y.sum()
            )

            negative_n = int(
                subgroup_n
                - positive_n
            )

            stability_warning = (
                subgroup_stability_warning(
                    n=subgroup_n,
                    positive_n=positive_n,
                    negative_n=negative_n,
                )
            )

            for model_name in MODEL_NAMES:
                probability_column = (
                    "probability__"
                    + model_name
                    + "__"
                    + target
                )

                probability = (
                    subgroup_data[
                        probability_column
                    ].to_numpy(
                        dtype=float
                    )
                )

                probability_metrics = (
                    calculate_probability_metrics(
                        y,
                        probability,
                    )
                )

                for metric in PROBABILITY_METRICS:
                    absolute_point_rows.append(
                        {
                            "model": model_name,
                            "model_display_name": (
                                MODEL_DISPLAY_NAMES[
                                    model_name
                                ]
                            ),
                            "target": target,
                            "target_display_name": (
                                TARGET_DISPLAY_NAMES[
                                    target
                                ]
                            ),
                            "target_training_sensitivity": np.nan,
                            "subgroup_variable": (
                                subgroup_variable
                            ),
                            "subgroup_variable_display_name": (
                                SUBGROUP_DISPLAY_NAMES[
                                    subgroup_variable
                                ]
                            ),
                            "subgroup": subgroup,
                            "reference_group": (
                                REFERENCE_GROUPS[
                                    subgroup_variable
                                ]
                            ),
                            "is_reference_group": bool(
                                subgroup
                                == REFERENCE_GROUPS[
                                    subgroup_variable
                                ]
                            ),
                            "n": subgroup_n,
                            "target_positive_n": (
                                positive_n
                            ),
                            "target_negative_n": (
                                negative_n
                            ),
                            "stability_warning": (
                                stability_warning
                            ),
                            "metric": metric,
                            "metric_display_name": (
                                METRIC_DISPLAY_NAMES[
                                    metric
                                ]
                            ),
                            "target_specific_metric_label": (
                                target_specific_metric_label(
                                    metric,
                                    target,
                                )
                            ),
                            "metric_depends_on_threshold": False,
                            "estimate": (
                                probability_metrics[
                                    metric
                                ]
                            ),
                            "calibration_fit_success": (
                                probability_metrics[
                                    "calibration_fit_success"
                                ]
                            ),
                        }
                    )

                for (
                    target_sensitivity
                ) in TARGET_SENSITIVITY_LEVELS:
                    sensitivity_key = str(
                        int(
                            round(
                                100
                                * target_sensitivity
                            )
                        )
                    )

                    prediction_column = (
                        "predicted_class__"
                        + model_name
                        + "__"
                        + target
                        + "__sens_"
                        + sensitivity_key
                    )

                    predicted_class = (
                        subgroup_data[
                            prediction_column
                        ].to_numpy(
                            dtype=int
                        )
                    )

                    threshold_metrics = (
                        calculate_threshold_metrics(
                            y,
                            predicted_class,
                        )
                    )

                    for metric in THRESHOLD_METRICS:
                        absolute_point_rows.append(
                            {
                                "model": model_name,
                                "model_display_name": (
                                    MODEL_DISPLAY_NAMES[
                                        model_name
                                    ]
                                ),
                                "target": target,
                                "target_display_name": (
                                    TARGET_DISPLAY_NAMES[
                                        target
                                    ]
                                ),
                                "target_training_sensitivity": float(
                                    target_sensitivity
                                ),
                                "subgroup_variable": (
                                    subgroup_variable
                                ),
                                "subgroup_variable_display_name": (
                                    SUBGROUP_DISPLAY_NAMES[
                                        subgroup_variable
                                    ]
                                ),
                                "subgroup": subgroup,
                                "reference_group": (
                                    REFERENCE_GROUPS[
                                        subgroup_variable
                                    ]
                                ),
                                "is_reference_group": bool(
                                    subgroup
                                    == REFERENCE_GROUPS[
                                        subgroup_variable
                                    ]
                                ),
                                "n": subgroup_n,
                                "target_positive_n": (
                                    positive_n
                                ),
                                "target_negative_n": (
                                    negative_n
                                ),
                                "stability_warning": (
                                    stability_warning
                                ),
                                "metric": metric,
                                "metric_display_name": (
                                    METRIC_DISPLAY_NAMES[
                                        metric
                                    ]
                                ),
                                "target_specific_metric_label": (
                                    target_specific_metric_label(
                                        metric,
                                        target,
                                    )
                                ),
                                "metric_depends_on_threshold": True,
                                "estimate": (
                                    threshold_metrics[
                                        metric
                                    ]
                                ),
                                "calibration_fit_success": np.nan,
                            }
                        )

absolute_point_estimates = (
    pd.DataFrame(
        absolute_point_rows
    )
)

absolute_point_estimates.to_csv(
    TABLE_DIR
    / "fairness_absolute_subgroup_metrics_point_estimates.csv",
    index=False,
)

absolute_point_estimates


## 10. Point estimates for reference-group gaps

In [ ]:
def create_reference_gap_table(
    absolute_table: pd.DataFrame,
) -> pd.DataFrame:
    result_rows = []

    grouping_columns = [
        "model",
        "target",
        "metric",
        "metric_depends_on_threshold",
        "subgroup_variable",
    ]

    threshold_values = [
        *TARGET_SENSITIVITY_LEVELS,
        np.nan,
    ]

    for (
        model_name,
        target,
        metric,
        metric_depends_on_threshold,
        subgroup_variable,
    ), group in absolute_table.groupby(
        grouping_columns,
        dropna=False,
        observed=True,
    ):
        if metric_depends_on_threshold:
            relevant_thresholds = (
                TARGET_SENSITIVITY_LEVELS
            )
        else:
            relevant_thresholds = [
                np.nan
            ]

        for threshold_value in relevant_thresholds:
            if pd.isna(
                threshold_value
            ):
                threshold_group = (
                    group.loc[
                        group[
                            "target_training_sensitivity"
                        ].isna()
                    ]
                )
            else:
                threshold_group = (
                    group.loc[
                        np.isclose(
                            group[
                                "target_training_sensitivity"
                            ],
                            threshold_value,
                        )
                    ]
                )

            reference_group = (
                REFERENCE_GROUPS[
                    subgroup_variable
                ]
            )

            reference_rows = (
                threshold_group.loc[
                    threshold_group[
                        "subgroup"
                    ]
                    == reference_group
                ]
            )

            if len(
                reference_rows
            ) != 1:
                raise ValueError(
                    "Expected exactly one reference-group row for "
                    f"{model_name}, {target}, {metric}, "
                    f"{subgroup_variable}, {threshold_value}."
                )

            reference_estimate = float(
                reference_rows[
                    "estimate"
                ].iloc[0]
            )

            for _, row in (
                threshold_group
                .iterrows()
            ):
                result_rows.append(
                    {
                        **row.to_dict(),
                        "reference_group_estimate": (
                            reference_estimate
                        ),
                        "gap_subgroup_minus_reference": (
                            float(
                                row[
                                    "estimate"
                                ]
                                - reference_estimate
                            )
                            if (
                                not pd.isna(
                                    row[
                                        "estimate"
                                    ]
                                )
                                and not pd.isna(
                                    reference_estimate
                                )
                            )
                            else np.nan
                        ),
                    }
                )

    return pd.DataFrame(
        result_rows
    )


reference_gap_point_estimates = (
    create_reference_gap_table(
        absolute_point_estimates
    )
)

reference_gap_point_estimates.to_csv(
    TABLE_DIR
    / "fairness_reference_gaps_point_estimates.csv",
    index=False,
)

reference_gap_point_estimates


## 11. Point estimates for target-dependent differences in subgroup gaps

For each model, subgroup, metric, and operating point, the difference is

$$
\left(
\text{subgroup-reference gap under HbA1c}
\right)
-
\left(
\text{subgroup-reference gap under prior diagnosis}
\right).
$$

A nonzero value indicates that the observed subgroup gap changes when the operational target changes.

In [ ]:
cross_target_gap_index = [
    "model",
    "metric",
    "metric_depends_on_threshold",
    "target_training_sensitivity",
    "subgroup_variable",
    "subgroup",
    "reference_group",
    "is_reference_group",
]

cross_target_gap_wide = (
    reference_gap_point_estimates
    .pivot(
        index=(
            cross_target_gap_index
        ),
        columns="target",
        values=(
            "gap_subgroup_minus_reference"
        ),
    )
    .reset_index()
)

cross_target_gap_wide.columns.name = None

cross_target_gap_wide[
    "difference_hba1c_minus_prior_gap"
] = (
    cross_target_gap_wide[
        "current_hba1c_ge_6_5"
    ]
    - cross_target_gap_wide[
        "self_reported_prior_diagnosis"
    ]
)

cross_target_gap_wide[
    "model_display_name"
] = (
    cross_target_gap_wide[
        "model"
    ].map(
        MODEL_DISPLAY_NAMES
    )
)

cross_target_gap_wide[
    "metric_display_name"
] = (
    cross_target_gap_wide[
        "metric"
    ].map(
        METRIC_DISPLAY_NAMES
    )
)

cross_target_gap_wide.to_csv(
    TABLE_DIR
    / "fairness_cross_target_gap_differences_point_estimates.csv",
    index=False,
)

cross_target_gap_wide


## Bootstrap uncertainty

## 12. Prepare efficient participant-level arrays

In [ ]:
participant_n = int(
    len(
        participant_analysis
    )
)

target_arrays = {
    target: (
        participant_analysis[
            target
        ].to_numpy(
            dtype=int
        )
    )
    for target in TARGET_COLUMNS
}

probability_arrays = {
    (
        model_name,
        target,
    ): (
        participant_analysis[
            (
                "probability__"
                + model_name
                + "__"
                + target
            )
        ].to_numpy(
            dtype=float
        )
    )
    for model_name in MODEL_NAMES
    for target in TARGET_COLUMNS
}

prediction_arrays = {
    (
        model_name,
        target,
        target_sensitivity,
    ): (
        participant_analysis[
            (
                "predicted_class__"
                + model_name
                + "__"
                + target
                + "__sens_"
                + str(
                    int(
                        round(
                            100
                            * target_sensitivity
                        )
                    )
                )
            )
        ].to_numpy(
            dtype=int
        )
    )
    for model_name in MODEL_NAMES
    for target in TARGET_COLUMNS
    for target_sensitivity
    in TARGET_SENSITIVITY_LEVELS
}

subgroup_arrays = {
    subgroup_variable: (
        participant_analysis[
            subgroup_variable
        ].astype(str).to_numpy()
    )
    for subgroup_variable
    in SUBGROUP_LEVELS
}

print(
    "Prepared participant-level arrays for bootstrap."
)


## 13. Run the paired participant-level bootstrap

The bootstrap stores only the metric values needed for confidence-interval summaries. It does not write a very large replicate-level CSV.

In [ ]:
absolute_bootstrap_values = defaultdict(
    list
)

gap_bootstrap_values = defaultdict(
    list
)

cross_target_gap_bootstrap_values = (
    defaultdict(list)
)

bootstrap_rng = np.random.default_rng(
    RANDOM_STATE
)

bootstrap_start_time = time.time()

for replicate in range(
    1,
    N_BOOTSTRAP + 1,
):
    sampled_indices = (
        bootstrap_rng.integers(
            low=0,
            high=participant_n,
            size=participant_n,
        )
    )

    replicate_absolute = {}

    for (
        subgroup_variable,
        subgroup_levels,
    ) in SUBGROUP_LEVELS.items():
        sampled_subgroup_values = (
            subgroup_arrays[
                subgroup_variable
            ][
                sampled_indices
            ]
        )

        for subgroup in subgroup_levels:
            subgroup_positions = (
                sampled_indices[
                    sampled_subgroup_values
                    == subgroup
                ]
            )

            subgroup_n = int(
                subgroup_positions.size
            )

            if subgroup_n == 0:
                continue

            for target in TARGET_COLUMNS:
                y = (
                    target_arrays[
                        target
                    ][
                        subgroup_positions
                    ]
                )

                for model_name in MODEL_NAMES:
                    probability = (
                        probability_arrays[
                            (
                                model_name,
                                target,
                            )
                        ][
                            subgroup_positions
                        ]
                    )

                    probability_metrics = (
                        calculate_probability_metrics(
                            y,
                            probability,
                        )
                    )

                    for metric in PROBABILITY_METRICS:
                        key = (
                            model_name,
                            target,
                            np.nan,
                            subgroup_variable,
                            subgroup,
                            metric,
                        )

                        value = (
                            probability_metrics[
                                metric
                            ]
                        )

                        absolute_bootstrap_values[
                            key
                        ].append(
                            value
                        )

                        replicate_absolute[
                            key
                        ] = value

                    for (
                        target_sensitivity
                    ) in TARGET_SENSITIVITY_LEVELS:
                        predicted_class = (
                            prediction_arrays[
                                (
                                    model_name,
                                    target,
                                    target_sensitivity,
                                )
                            ][
                                subgroup_positions
                            ]
                        )

                        threshold_metrics = (
                            calculate_threshold_metrics(
                                y,
                                predicted_class,
                            )
                        )

                        for metric in THRESHOLD_METRICS:
                            key = (
                                model_name,
                                target,
                                float(
                                    target_sensitivity
                                ),
                                subgroup_variable,
                                subgroup,
                                metric,
                            )

                            value = (
                                threshold_metrics[
                                    metric
                                ]
                            )

                            absolute_bootstrap_values[
                                key
                            ].append(
                                value
                            )

                            replicate_absolute[
                                key
                            ] = value

    # Reference-group gaps within each target.
    replicate_gaps = {}

    for (
        model_name,
        target,
        threshold_value,
        subgroup_variable,
        subgroup,
        metric,
    ), subgroup_value in (
        replicate_absolute.items()
    ):
        reference_group = (
            REFERENCE_GROUPS[
                subgroup_variable
            ]
        )

        reference_key = (
            model_name,
            target,
            threshold_value,
            subgroup_variable,
            reference_group,
            metric,
        )

        reference_value = (
            replicate_absolute.get(
                reference_key,
                np.nan,
            )
        )

        gap_key = (
            model_name,
            target,
            threshold_value,
            subgroup_variable,
            subgroup,
            metric,
        )

        if (
            pd.isna(
                subgroup_value
            )
            or pd.isna(
                reference_value
            )
        ):
            gap_value = np.nan
        else:
            gap_value = float(
                subgroup_value
                - reference_value
            )

        gap_bootstrap_values[
            gap_key
        ].append(
            gap_value
        )

        replicate_gaps[
            gap_key
        ] = gap_value

    # HbA1c-target gap minus prior-diagnosis-target gap.
    for model_name in MODEL_NAMES:
        for (
            subgroup_variable,
            subgroup_levels,
        ) in SUBGROUP_LEVELS.items():
            for subgroup in subgroup_levels:
                for metric in ALL_METRICS:
                    if (
                        METRIC_DEPENDS_ON_THRESHOLD[
                            metric
                        ]
                    ):
                        relevant_thresholds = (
                            TARGET_SENSITIVITY_LEVELS
                        )
                    else:
                        relevant_thresholds = [
                            np.nan
                        ]

                    for threshold_value in relevant_thresholds:
                        prior_key = (
                            model_name,
                            "self_reported_prior_diagnosis",
                            threshold_value,
                            subgroup_variable,
                            subgroup,
                            metric,
                        )

                        hba1c_key = (
                            model_name,
                            "current_hba1c_ge_6_5",
                            threshold_value,
                            subgroup_variable,
                            subgroup,
                            metric,
                        )

                        prior_gap = (
                            replicate_gaps.get(
                                prior_key,
                                np.nan,
                            )
                        )

                        hba1c_gap = (
                            replicate_gaps.get(
                                hba1c_key,
                                np.nan,
                            )
                        )

                        cross_target_key = (
                            model_name,
                            threshold_value,
                            subgroup_variable,
                            subgroup,
                            metric,
                        )

                        if (
                            pd.isna(
                                prior_gap
                            )
                            or pd.isna(
                                hba1c_gap
                            )
                        ):
                            difference = np.nan
                        else:
                            difference = float(
                                hba1c_gap
                                - prior_gap
                            )

                        cross_target_gap_bootstrap_values[
                            cross_target_key
                        ].append(
                            difference
                        )

    if (
        replicate
        % BOOTSTRAP_PROGRESS_EVERY
        == 0
        or replicate == N_BOOTSTRAP
    ):
        elapsed_minutes = (
            time.time()
            - bootstrap_start_time
        ) / 60

        print(
            f"Completed {replicate:,}/{N_BOOTSTRAP:,} "
            f"bootstrap replicates "
            f"({elapsed_minutes:.1f} minutes elapsed)."
        )

print("Bootstrap completed.")


## 14. Convert bootstrap distributions into confidence intervals

In [ ]:
def summarise_bootstrap_values(
    values,
) -> dict:
    array = np.asarray(
        values,
        dtype=float,
    )

    finite_values = (
        array[
            np.isfinite(
                array
            )
        ]
    )

    if finite_values.size == 0:
        return {
            "bootstrap_mean": np.nan,
            "bootstrap_standard_error": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "successful_bootstrap_replicates": 0,
        }

    return {
        "bootstrap_mean": float(
            np.mean(
                finite_values
            )
        ),
        "bootstrap_standard_error": float(
            np.std(
                finite_values,
                ddof=1,
            )
        )
        if finite_values.size > 1
        else np.nan,
        "ci_lower": float(
            np.quantile(
                finite_values,
                CI_LOWER_QUANTILE,
            )
        ),
        "ci_upper": float(
            np.quantile(
                finite_values,
                CI_UPPER_QUANTILE,
            )
        ),
        "successful_bootstrap_replicates": int(
            finite_values.size
        ),
    }


absolute_bootstrap_rows = []

for key, values in (
    absolute_bootstrap_values.items()
):
    (
        model_name,
        target,
        threshold_value,
        subgroup_variable,
        subgroup,
        metric,
    ) = key

    absolute_bootstrap_rows.append(
        {
            "model": model_name,
            "target": target,
            "target_training_sensitivity": (
                threshold_value
            ),
            "subgroup_variable": (
                subgroup_variable
            ),
            "subgroup": subgroup,
            "metric": metric,
            **summarise_bootstrap_values(
                values
            ),
        }
    )

absolute_bootstrap_summary = (
    pd.DataFrame(
        absolute_bootstrap_rows
    )
)

gap_bootstrap_rows = []

for key, values in (
    gap_bootstrap_values.items()
):
    (
        model_name,
        target,
        threshold_value,
        subgroup_variable,
        subgroup,
        metric,
    ) = key

    summary = (
        summarise_bootstrap_values(
            values
        )
    )

    gap_bootstrap_rows.append(
        {
            "model": model_name,
            "target": target,
            "target_training_sensitivity": (
                threshold_value
            ),
            "subgroup_variable": (
                subgroup_variable
            ),
            "subgroup": subgroup,
            "metric": metric,
            "gap_bootstrap_mean": (
                summary[
                    "bootstrap_mean"
                ]
            ),
            "gap_bootstrap_standard_error": (
                summary[
                    "bootstrap_standard_error"
                ]
            ),
            "gap_ci_lower": (
                summary[
                    "ci_lower"
                ]
            ),
            "gap_ci_upper": (
                summary[
                    "ci_upper"
                ]
            ),
            "gap_successful_bootstrap_replicates": (
                summary[
                    "successful_bootstrap_replicates"
                ]
            ),
        }
    )

gap_bootstrap_summary = pd.DataFrame(
    gap_bootstrap_rows
)

cross_target_bootstrap_rows = []

for key, values in (
    cross_target_gap_bootstrap_values.items()
):
    (
        model_name,
        threshold_value,
        subgroup_variable,
        subgroup,
        metric,
    ) = key

    summary = (
        summarise_bootstrap_values(
            values
        )
    )

    cross_target_bootstrap_rows.append(
        {
            "model": model_name,
            "target_training_sensitivity": (
                threshold_value
            ),
            "subgroup_variable": (
                subgroup_variable
            ),
            "subgroup": subgroup,
            "metric": metric,
            "difference_bootstrap_mean": (
                summary[
                    "bootstrap_mean"
                ]
            ),
            "difference_bootstrap_standard_error": (
                summary[
                    "bootstrap_standard_error"
                ]
            ),
            "difference_ci_lower": (
                summary[
                    "ci_lower"
                ]
            ),
            "difference_ci_upper": (
                summary[
                    "ci_upper"
                ]
            ),
            "difference_successful_bootstrap_replicates": (
                summary[
                    "successful_bootstrap_replicates"
                ]
            ),
        }
    )

cross_target_bootstrap_summary = (
    pd.DataFrame(
        cross_target_bootstrap_rows
    )
)

print(
    "Absolute bootstrap summaries:",
    absolute_bootstrap_summary.shape,
)
print(
    "Gap bootstrap summaries:",
    gap_bootstrap_summary.shape,
)
print(
    "Cross-target gap summaries:",
    cross_target_bootstrap_summary.shape,
)


## 15. Combine point estimates with bootstrap intervals

In [ ]:
merge_keys = [
    "model",
    "target",
    "target_training_sensitivity",
    "subgroup_variable",
    "subgroup",
    "metric",
]

absolute_metrics_with_intervals = (
    absolute_point_estimates
    .merge(
        absolute_bootstrap_summary,
        on=merge_keys,
        how="left",
        validate="one_to_one",
    )
)

absolute_metrics_with_intervals[
    "bootstrap_replicates_requested"
] = N_BOOTSTRAP

absolute_metrics_with_intervals.to_csv(
    TABLE_DIR
    / "fairness_absolute_subgroup_metrics_with_intervals.csv",
    index=False,
)

reference_gaps_with_intervals = (
    reference_gap_point_estimates
    .merge(
        absolute_bootstrap_summary,
        on=merge_keys,
        how="left",
        validate="one_to_one",
    )
    .merge(
        gap_bootstrap_summary,
        on=merge_keys,
        how="left",
        validate="one_to_one",
    )
)

reference_gaps_with_intervals[
    "bootstrap_replicates_requested"
] = N_BOOTSTRAP

reference_gaps_with_intervals.to_csv(
    TABLE_DIR
    / "fairness_reference_gaps_with_intervals.csv",
    index=False,
)

cross_target_merge_keys = [
    "model",
    "target_training_sensitivity",
    "subgroup_variable",
    "subgroup",
    "metric",
]

cross_target_gaps_with_intervals = (
    cross_target_gap_wide
    .merge(
        cross_target_bootstrap_summary,
        on=(
            cross_target_merge_keys
        ),
        how="left",
        validate="one_to_one",
    )
)

cross_target_gaps_with_intervals[
    "bootstrap_replicates_requested"
] = N_BOOTSTRAP

cross_target_gaps_with_intervals.to_csv(
    TABLE_DIR
    / "fairness_cross_target_gap_differences_with_intervals.csv",
    index=False,
)

display(
    absolute_metrics_with_intervals.head()
)

display(
    reference_gaps_with_intervals.head()
)

display(
    cross_target_gaps_with_intervals.head()
)


## 16. Check bootstrap coverage and calibration estimability

In [ ]:
bootstrap_coverage_summary = (
    absolute_metrics_with_intervals
    .groupby(
        "metric",
        as_index=False,
    )
    .agg(
        rows=(
            "metric",
            "size",
        ),
        minimum_successful_bootstrap_replicates=(
            "successful_bootstrap_replicates",
            "min",
        ),
        median_successful_bootstrap_replicates=(
            "successful_bootstrap_replicates",
            "median",
        ),
        maximum_successful_bootstrap_replicates=(
            "successful_bootstrap_replicates",
            "max",
        ),
    )
)

bootstrap_coverage_summary[
    "requested_bootstrap_replicates"
] = N_BOOTSTRAP

bootstrap_coverage_summary.to_csv(
    TABLE_DIR
    / "fairness_bootstrap_coverage_summary.csv",
    index=False,
)

calibration_estimability = (
    absolute_point_estimates
    .loc[
        absolute_point_estimates[
            "metric"
        ].isin(
            [
                "calibration_intercept",
                "calibration_slope",
            ]
        )
    ]
    [
        [
            "model",
            "target",
            "subgroup_variable",
            "subgroup",
            "n",
            "target_positive_n",
            "target_negative_n",
            "stability_warning",
            "calibration_fit_success",
        ]
    ]
    .drop_duplicates()
)

calibration_estimability.to_csv(
    TABLE_DIR
    / "fairness_calibration_estimability.csv",
    index=False,
)

display(
    bootstrap_coverage_summary
)

display(
    calibration_estimability
)


## Table 4 — Primary 80%-sensitivity subgroup metrics

## 17. Create the main long-format table

The primary table contains:

- all threshold-dependent metrics at the 80% training-sensitivity operating point;
- Brier score and calibration metrics from the same out-of-fold probabilities;
- absolute subgroup estimates and confidence intervals;
- reference-group gaps and confidence intervals;
- counts and stability warnings.

In [ ]:
primary_threshold_rows = (
    reference_gaps_with_intervals
    .loc[
        (
            reference_gaps_with_intervals[
                "metric_depends_on_threshold"
            ]
            & np.isclose(
                reference_gaps_with_intervals[
                    "target_training_sensitivity"
                ],
                PRIMARY_TARGET_SENSITIVITY,
            )
        )
        | (
            ~reference_gaps_with_intervals[
                "metric_depends_on_threshold"
            ]
            & reference_gaps_with_intervals[
                "target_training_sensitivity"
            ].isna()
        )
    ]
    .copy()
)

primary_threshold_rows[
    "primary_operating_point"
] = (
    PRIMARY_TARGET_SENSITIVITY
)

primary_threshold_rows[
    "estimate_percentage"
] = np.where(
    primary_threshold_rows[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_threshold_rows[
        "estimate"
    ],
    np.nan,
)

primary_threshold_rows[
    "ci_lower_percentage"
] = np.where(
    primary_threshold_rows[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_threshold_rows[
        "ci_lower"
    ],
    np.nan,
)

primary_threshold_rows[
    "ci_upper_percentage"
] = np.where(
    primary_threshold_rows[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_threshold_rows[
        "ci_upper"
    ],
    np.nan,
)

primary_threshold_rows[
    "gap_percentage_points"
] = np.where(
    primary_threshold_rows[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_threshold_rows[
        "gap_subgroup_minus_reference"
    ],
    np.nan,
)

primary_threshold_rows[
    "gap_ci_lower_percentage_points"
] = np.where(
    primary_threshold_rows[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_threshold_rows[
        "gap_ci_lower"
    ],
    np.nan,
)

primary_threshold_rows[
    "gap_ci_upper_percentage_points"
] = np.where(
    primary_threshold_rows[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_threshold_rows[
        "gap_ci_upper"
    ],
    np.nan,
)

primary_threshold_rows.to_csv(
    TABLE_DIR
    / "table4_primary_subgroup_metrics_long.csv",
    index=False,
)

print(
    "Table 4 long rows:",
    len(
        primary_threshold_rows
    ),
)

primary_threshold_rows.head(20)


## 18. Create a paper-ready wide version of Table 4

In [ ]:
def format_interval(
    estimate: float,
    lower: float,
    upper: float,
    digits: int = 3,
    multiplier: float = 1.0,
) -> str:
    if (
        pd.isna(
            estimate
        )
        or pd.isna(
            lower
        )
        or pd.isna(
            upper
        )
    ):
        return (
            "Not estimable"
        )

    return (
        f"{multiplier * estimate:.{digits}f} "
        f"[{multiplier * lower:.{digits}f}, "
        f"{multiplier * upper:.{digits}f}]"
    )


def format_gap_interval(
    estimate: float,
    lower: float,
    upper: float,
    digits: int = 3,
    multiplier: float = 1.0,
) -> str:
    if (
        pd.isna(
            estimate
        )
        or pd.isna(
            lower
        )
        or pd.isna(
            upper
        )
    ):
        return (
            "Not estimable"
        )

    return (
        f"{multiplier * estimate:+.{digits}f} "
        f"[{multiplier * lower:+.{digits}f}, "
        f"{multiplier * upper:+.{digits}f}]"
    )


table4_formatted = (
    primary_threshold_rows
    .copy()
)

table4_formatted[
    "absolute_estimate_95_ci"
] = table4_formatted.apply(
    lambda row:
    format_interval(
        estimate=row[
            "estimate"
        ],
        lower=row[
            "ci_lower"
        ],
        upper=row[
            "ci_upper"
        ],
        digits=1
        if row["metric"]
        in THRESHOLD_METRICS
        else 3,
        multiplier=100
        if row["metric"]
        in THRESHOLD_METRICS
        else 1,
    ),
    axis=1,
)

table4_formatted[
    "gap_vs_reference_95_ci"
] = table4_formatted.apply(
    lambda row:
    format_gap_interval(
        estimate=row[
            "gap_subgroup_minus_reference"
        ],
        lower=row[
            "gap_ci_lower"
        ],
        upper=row[
            "gap_ci_upper"
        ],
        digits=1
        if row["metric"]
        in THRESHOLD_METRICS
        else 3,
        multiplier=100
        if row["metric"]
        in THRESHOLD_METRICS
        else 1,
    ),
    axis=1,
)

table4_wide = (
    table4_formatted
    .pivot(
        index=[
            "model",
            "model_display_name",
            "target",
            "target_display_name",
            "subgroup_variable",
            "subgroup_variable_display_name",
            "subgroup",
            "reference_group",
            "is_reference_group",
            "n",
            "target_positive_n",
            "target_negative_n",
            "stability_warning",
        ],
        columns="metric",
        values=[
            "absolute_estimate_95_ci",
            "gap_vs_reference_95_ci",
        ],
    )
    .reset_index()
)

table4_wide.columns = [
    (
        column
        if isinstance(
            column,
            str,
        )
        else "__".join(
            str(part)
            for part
            in column
            if str(part) != ""
        )
    )
    for column
    in table4_wide.columns
]

table4_wide.to_csv(
    TABLE_DIR
    / "table4_primary_subgroup_metrics_wide.csv",
    index=False,
)

table4_wide.head()


## 19. Primary cross-target gap-difference table

In [ ]:
primary_cross_target_gap_differences = (
    cross_target_gaps_with_intervals
    .loc[
        (
            cross_target_gaps_with_intervals[
                "metric_depends_on_threshold"
            ]
            & np.isclose(
                cross_target_gaps_with_intervals[
                    "target_training_sensitivity"
                ],
                PRIMARY_TARGET_SENSITIVITY,
            )
        )
        | (
            ~cross_target_gaps_with_intervals[
                "metric_depends_on_threshold"
            ]
            & cross_target_gaps_with_intervals[
                "target_training_sensitivity"
            ].isna()
        )
    ]
    .copy()
)

primary_cross_target_gap_differences[
    "difference_percentage_points"
] = np.where(
    primary_cross_target_gap_differences[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_cross_target_gap_differences[
        "difference_hba1c_minus_prior_gap"
    ],
    np.nan,
)

primary_cross_target_gap_differences[
    "difference_ci_lower_percentage_points"
] = np.where(
    primary_cross_target_gap_differences[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_cross_target_gap_differences[
        "difference_ci_lower"
    ],
    np.nan,
)

primary_cross_target_gap_differences[
    "difference_ci_upper_percentage_points"
] = np.where(
    primary_cross_target_gap_differences[
        "metric"
    ].isin(
        THRESHOLD_METRICS
    ),
    100
    * primary_cross_target_gap_differences[
        "difference_ci_upper"
    ],
    np.nan,
)

primary_cross_target_gap_differences.to_csv(
    TABLE_DIR
    / "table4_cross_target_gap_differences_primary.csv",
    index=False,
)

primary_cross_target_gap_differences.head(20)


## Figure 4 — Subgroup false-negative rates

## 20. Prepare the primary 80%-sensitivity FNR figure

The figure includes:

- both models;
- both target definitions;
- all prespecified subgroup categories;
- participant-bootstrap confidence intervals.

The false-negative rate is always interpreted relative to the displayed target.

In [ ]:
figure4_data = (
    absolute_metrics_with_intervals
    .loc[
        (
            absolute_metrics_with_intervals[
                "metric"
            ]
            == "false_negative_rate"
        )
        & np.isclose(
            absolute_metrics_with_intervals[
                "target_training_sensitivity"
            ],
            PRIMARY_TARGET_SENSITIVITY,
        )
    ]
    .copy()
)

subgroup_plot_order = [
    (
        subgroup_variable,
        subgroup,
    )
    for subgroup_variable, levels
    in SUBGROUP_LEVELS.items()
    for subgroup in levels
]

figure4_data[
    "plot_key"
] = list(
    zip(
        figure4_data[
            "subgroup_variable"
        ],
        figure4_data[
            "subgroup"
        ],
    )
)

figure4_data[
    "plot_key"
] = pd.Categorical(
    figure4_data[
        "plot_key"
    ],
    categories=(
        subgroup_plot_order
    ),
    ordered=True,
)

figure4_data = (
    figure4_data
    .sort_values(
        [
            "plot_key",
            "target",
            "model",
        ]
    )
)

plot_label_lookup = {
    (
        subgroup_variable,
        subgroup,
    ): (
        SUBGROUP_DISPLAY_NAMES[
            subgroup_variable
        ]
        + " — "
        + subgroup
    )
    for (
        subgroup_variable,
        subgroup,
    ) in subgroup_plot_order
}

series_order = [
    (
        "logistic",
        "self_reported_prior_diagnosis",
    ),
    (
        "ebm",
        "self_reported_prior_diagnosis",
    ),
    (
        "logistic",
        "current_hba1c_ge_6_5",
    ),
    (
        "ebm",
        "current_hba1c_ge_6_5",
    ),
]

series_offsets = {
    series: offset
    for series, offset
    in zip(
        series_order,
        [
            -0.24,
            -0.08,
            0.08,
            0.24,
        ],
    )
}

series_markers = {
    (
        "logistic",
        "self_reported_prior_diagnosis",
    ): "o",
    (
        "ebm",
        "self_reported_prior_diagnosis",
    ): "s",
    (
        "logistic",
        "current_hba1c_ge_6_5",
    ): "^",
    (
        "ebm",
        "current_hba1c_ge_6_5",
    ): "D",
}

vertical_positions = np.arange(
    len(
        subgroup_plot_order
    )
)

figure, axis = plt.subplots(
    figsize=(
        12,
        10,
    )
)

for (
    model_name,
    target,
) in series_order:
    series_data = (
        figure4_data
        .loc[
            (
                figure4_data[
                    "model"
                ]
                == model_name
            )
            & (
                figure4_data[
                    "target"
                ]
                == target
            )
        ]
        .set_index(
            "plot_key"
        )
        .reindex(
            subgroup_plot_order
        )
    )

    estimates = (
        100
        * series_data[
            "estimate"
        ].to_numpy(
            dtype=float
        )
    )

    interval_lower = (
        100
        * series_data[
            "ci_lower"
        ].to_numpy(
            dtype=float
        )
    )

    interval_upper = (
        100
        * series_data[
            "ci_upper"
        ].to_numpy(
            dtype=float
        )
    )

    positions = (
        vertical_positions
        + series_offsets[
            (
                model_name,
                target,
            )
        ]
    )

    plotted = axis.plot(
        estimates,
        positions,
        linestyle="none",
        marker=(
            series_markers[
                (
                    model_name,
                    target,
                )
            ]
        ),
        label=(
            MODEL_DISPLAY_NAMES[
                model_name
            ]
            + " — "
            + TARGET_SHORT_NAMES[
                target
            ]
        ),
    )

    automatic_colour = (
        plotted[0]
        .get_color()
    )

    axis.hlines(
        positions,
        interval_lower,
        interval_upper,
        color=automatic_colour,
    )

axis.set_yticks(
    vertical_positions,
    labels=[
        plot_label_lookup[
            plot_key
        ]
        for plot_key
        in subgroup_plot_order
    ],
)

axis.set_xlabel(
    "False-negative rate at the primary operating point (%)"
)

axis.set_ylabel(
    "Evaluation subgroup"
)

axis.set_title(
    "Figure 4. Subgroup false-negative rates "
    "under two target definitions"
)

axis.set_xlim(
    left=0,
)

axis.legend(
    loc="best"
)

axis.invert_yaxis()

figure.tight_layout()

figure4_png_path = (
    FIGURE_DIR
    / "figure4_subgroup_false_negative_rates.png"
)

figure4_pdf_path = (
    FIGURE_DIR
    / "figure4_subgroup_false_negative_rates.pdf"
)

figure.savefig(
    figure4_png_path,
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure4_pdf_path,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:")
print(figure4_png_path)
print(figure4_pdf_path)


## 21. Appendix: subgroup FNR gaps from the reference group

In [ ]:
fnr_gap_data = (
    reference_gaps_with_intervals
    .loc[
        (
            reference_gaps_with_intervals[
                "metric"
            ]
            == "false_negative_rate"
        )
        & np.isclose(
            reference_gaps_with_intervals[
                "target_training_sensitivity"
            ],
            PRIMARY_TARGET_SENSITIVITY,
        )
    ]
    .copy()
)

fnr_gap_data[
    "plot_key"
] = list(
    zip(
        fnr_gap_data[
            "subgroup_variable"
        ],
        fnr_gap_data[
            "subgroup"
        ],
    )
)

fnr_gap_data[
    "plot_key"
] = pd.Categorical(
    fnr_gap_data[
        "plot_key"
    ],
    categories=(
        subgroup_plot_order
    ),
    ordered=True,
)

figure, axis = plt.subplots(
    figsize=(
        12,
        10,
    )
)

for (
    model_name,
    target,
) in series_order:
    series_data = (
        fnr_gap_data
        .loc[
            (
                fnr_gap_data[
                    "model"
                ]
                == model_name
            )
            & (
                fnr_gap_data[
                    "target"
                ]
                == target
            )
        ]
        .set_index(
            "plot_key"
        )
        .reindex(
            subgroup_plot_order
        )
    )

    estimates = (
        100
        * series_data[
            "gap_subgroup_minus_reference"
        ].to_numpy(
            dtype=float
        )
    )

    interval_lower = (
        100
        * series_data[
            "gap_ci_lower"
        ].to_numpy(
            dtype=float
        )
    )

    interval_upper = (
        100
        * series_data[
            "gap_ci_upper"
        ].to_numpy(
            dtype=float
        )
    )

    positions = (
        vertical_positions
        + series_offsets[
            (
                model_name,
                target,
            )
        ]
    )

    plotted = axis.plot(
        estimates,
        positions,
        linestyle="none",
        marker=(
            series_markers[
                (
                    model_name,
                    target,
                )
            ]
        ),
        label=(
            MODEL_DISPLAY_NAMES[
                model_name
            ]
            + " — "
            + TARGET_SHORT_NAMES[
                target
            ]
        ),
    )

    automatic_colour = (
        plotted[0]
        .get_color()
    )

    axis.hlines(
        positions,
        interval_lower,
        interval_upper,
        color=automatic_colour,
    )

axis.axvline(
    0,
    linewidth=1,
    linestyle="--",
)

axis.set_yticks(
    vertical_positions,
    labels=[
        plot_label_lookup[
            plot_key
        ]
        for plot_key
        in subgroup_plot_order
    ],
)

axis.set_xlabel(
    "False-negative-rate gap versus the prespecified "
    "reference group (percentage points)"
)

axis.set_ylabel(
    "Evaluation subgroup"
)

axis.set_title(
    "Appendix. Subgroup FNR gaps from reference groups"
)

axis.legend(
    loc="best"
)

axis.invert_yaxis()

figure.tight_layout()

fnr_gap_png_path = (
    FIGURE_DIR
    / "appendix_subgroup_fnr_reference_gaps.png"
)

fnr_gap_pdf_path = (
    FIGURE_DIR
    / "appendix_subgroup_fnr_reference_gaps.pdf"
)

figure.savefig(
    fnr_gap_png_path,
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    fnr_gap_pdf_path,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:")
print(fnr_gap_png_path)
print(fnr_gap_pdf_path)


## Interpretation aids

## 22. Identify the largest absolute primary FNR gaps

This table is a drafting aid only. It must be interpreted together with the confidence intervals and outcome counts.

In [ ]:
largest_fnr_gaps = (
    reference_gaps_with_intervals
    .loc[
        (
            reference_gaps_with_intervals[
                "metric"
            ]
            == "false_negative_rate"
        )
        & np.isclose(
            reference_gaps_with_intervals[
                "target_training_sensitivity"
            ],
            PRIMARY_TARGET_SENSITIVITY,
        )
        & (
            ~reference_gaps_with_intervals[
                "is_reference_group"
            ]
        )
    ]
    .assign(
        absolute_gap=(
            reference_gaps_with_intervals.loc[
                (
                    reference_gaps_with_intervals[
                        "metric"
                    ]
                    == "false_negative_rate"
                )
                & np.isclose(
                    reference_gaps_with_intervals[
                        "target_training_sensitivity"
                    ],
                    PRIMARY_TARGET_SENSITIVITY,
                )
                & (
                    ~reference_gaps_with_intervals[
                        "is_reference_group"
                    ]
                ),
                "gap_subgroup_minus_reference",
            ]
            .abs()
            .to_numpy()
        )
    )
    .sort_values(
        "absolute_gap",
        ascending=False,
    )
    [
        [
            "model_display_name",
            "target_display_name",
            "subgroup_variable_display_name",
            "subgroup",
            "reference_group",
            "n",
            "target_positive_n",
            "target_negative_n",
            "stability_warning",
            "estimate",
            "ci_lower",
            "ci_upper",
            "gap_subgroup_minus_reference",
            "gap_ci_lower",
            "gap_ci_upper",
        ]
    ]
)

largest_fnr_gaps.to_csv(
    TABLE_DIR
    / "fairness_largest_primary_fnr_gaps.csv",
    index=False,
)

largest_fnr_gaps.head(20)


## 23. Identify the largest target-dependent changes in FNR gaps

In [ ]:
largest_target_dependent_fnr_changes = (
    primary_cross_target_gap_differences
    .loc[
        (
            primary_cross_target_gap_differences[
                "metric"
            ]
            == "false_negative_rate"
        )
        & (
            ~primary_cross_target_gap_differences[
                "is_reference_group"
            ]
        )
    ]
    .assign(
        absolute_difference=(
            primary_cross_target_gap_differences.loc[
                (
                    primary_cross_target_gap_differences[
                        "metric"
                    ]
                    == "false_negative_rate"
                )
                & (
                    ~primary_cross_target_gap_differences[
                        "is_reference_group"
                    ]
                ),
                "difference_hba1c_minus_prior_gap",
            ]
            .abs()
            .to_numpy()
        )
    )
    .sort_values(
        "absolute_difference",
        ascending=False,
    )
)

largest_target_dependent_fnr_changes.to_csv(
    TABLE_DIR
    / "fairness_largest_target_dependent_fnr_gap_changes.csv",
    index=False,
)

largest_target_dependent_fnr_changes.head(20)


## Interpretation constraints

Report every subgroup result relative to its operational target, operating threshold, and reference group. Positive and negative gaps are descriptive model-performance differences; they do not by themselves establish fairness, discrimination, clinical benefit, or causal mechanisms.

## Save metadata and final checkpoint

In [ ]:
def json_safe(value):
    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(
                item
            )
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            json_safe(
                item
            )
            for item in value
        ]

    return value


output_files = {
    "subgroup_counts": (
        TABLE_DIR
        / "fairness_subgroup_counts.csv"
    ),
    "absolute_point_estimates": (
        TABLE_DIR
        / "fairness_absolute_subgroup_metrics_point_estimates.csv"
    ),
    "absolute_metrics_with_intervals": (
        TABLE_DIR
        / "fairness_absolute_subgroup_metrics_with_intervals.csv"
    ),
    "reference_gap_point_estimates": (
        TABLE_DIR
        / "fairness_reference_gaps_point_estimates.csv"
    ),
    "reference_gaps_with_intervals": (
        TABLE_DIR
        / "fairness_reference_gaps_with_intervals.csv"
    ),
    "cross_target_gap_point_estimates": (
        TABLE_DIR
        / "fairness_cross_target_gap_differences_point_estimates.csv"
    ),
    "cross_target_gaps_with_intervals": (
        TABLE_DIR
        / "fairness_cross_target_gap_differences_with_intervals.csv"
    ),
    "bootstrap_coverage_summary": (
        TABLE_DIR
        / "fairness_bootstrap_coverage_summary.csv"
    ),
    "calibration_estimability": (
        TABLE_DIR
        / "fairness_calibration_estimability.csv"
    ),
    "table4_long": (
        TABLE_DIR
        / "table4_primary_subgroup_metrics_long.csv"
    ),
    "table4_wide": (
        TABLE_DIR
        / "table4_primary_subgroup_metrics_wide.csv"
    ),
    "table4_cross_target_gap_differences": (
        TABLE_DIR
        / "table4_cross_target_gap_differences_primary.csv"
    ),
    "largest_primary_fnr_gaps": (
        TABLE_DIR
        / "fairness_largest_primary_fnr_gaps.csv"
    ),
    "largest_target_dependent_fnr_changes": (
        TABLE_DIR
        / "fairness_largest_target_dependent_fnr_gap_changes.csv"
    ),
    "figure4_png": (
        figure4_png_path
    ),
    "figure4_pdf": (
        figure4_pdf_path
    ),
    "appendix_fnr_gap_png": (
        fnr_gap_png_path
    ),
    "appendix_fnr_gap_pdf": (
        fnr_gap_pdf_path
    ),
}

missing_output_files = [
    str(path)
    for path
    in output_files.values()
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "The following expected Notebook 07 outputs are missing:\n"
        + "\n".join(
            missing_output_files
        )
    )

fairness_metadata = {
    "analytic_sample_n": int(
        len(
            participant_analysis
        )
    ),
    "analysis_id_sha256": (
        analysis_id_hash
    ),
    "random_state": (
        RANDOM_STATE
    ),
    "models": (
        MODEL_NAMES
    ),
    "all_primary_models_include_race_ethnicity": True,
    "race_ethnicity_roles": [
        "Primary model predictor",
        "Evaluation subgroup attribute",
    ],
    "targets": (
        TARGET_COLUMNS
    ),
    "subgroup_levels": (
        SUBGROUP_LEVELS
    ),
    "reference_groups": (
        REFERENCE_GROUPS
    ),
    "income_group_definition": {
        "Below 1": (
            "income-to-poverty ratio < 1"
        ),
        "1 to below 2": (
            "1 <= income-to-poverty ratio < 2"
        ),
        "2 to below 4": (
            "2 <= income-to-poverty ratio < 4"
        ),
        "4 or higher": (
            "income-to-poverty ratio >= 4"
        ),
    },
    "threshold_operating_points": (
        TARGET_SENSITIVITY_LEVELS
    ),
    "primary_operating_point": (
        PRIMARY_TARGET_SENSITIVITY
    ),
    "metrics": (
        ALL_METRICS
    ),
    "probability_metrics_are_threshold_independent": (
        PROBABILITY_METRICS
    ),
    "calibration_minimum_counts": {
        "minimum_n": (
            MIN_CALIBRATION_N
        ),
        "minimum_target_positive_n": (
            MIN_CALIBRATION_POSITIVE_N
        ),
        "minimum_target_negative_n": (
            MIN_CALIBRATION_NEGATIVE_N
        ),
    },
    "small_cell_warning_rules": {
        "very_unstable": (
            "fewer than 20 target-positive or target-negative participants"
        ),
        "caution": (
            "20 to 49 target-positive or target-negative participants"
        ),
    },
    "bootstrap": {
        "replicates": (
            N_BOOTSTRAP
        ),
        "resampling_unit": (
            "participant"
        ),
        "paired_across_models": True,
        "paired_across_targets": True,
        "paired_across_thresholds": True,
        "models_refit_inside_bootstrap": False,
        "thresholds_reselected_inside_bootstrap": False,
        "confidence_level": (
            CONFIDENCE_LEVEL
        ),
        "interval_method": (
            "percentile"
        ),
    },
    "cross_target_gap_difference": (
        "HbA1c subgroup-reference gap minus "
        "prior-diagnosis subgroup-reference gap"
    ),
    "race_categories_automatically_merged": False,
    "no_race_models_deferred_to_notebook08": True,
    "software_versions": {
        "python": (
            platform.python_version()
        ),
        "numpy": (
            np.__version__
        ),
        "pandas": (
            pd.__version__
        ),
        "scikit_learn": (
            sklearn.__version__
        ),
        "scipy": (
            __import__(
                "scipy"
            ).__version__
        ),
    },
    "output_files": {
        name: str(path)
        for name, path
        in output_files.items()
    },
}

with FAIRNESS_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            fairness_metadata
        ),
        file,
        indent=2,
    )

expected_absolute_rows = (
    len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
    * sum(
        len(levels)
        for levels
        in SUBGROUP_LEVELS.values()
    )
    * (
        len(PROBABILITY_METRICS)
        + len(
            TARGET_SENSITIVITY_LEVELS
        )
        * len(
            THRESHOLD_METRICS
        )
    )
)

final_checkpoint = {
    "analytic_sample_n": int(
        len(
            participant_analysis
        )
    ),
    "same_participants_as_notebook06": bool(
        performance_metadata[
            "analysis_id_sha256"
        ]
        == analysis_id_hash
    ),
    "four_primary_model_target_combinations": bool(
        absolute_point_estimates[
            [
                "model",
                "target",
            ]
        ]
        .drop_duplicates()
        .shape[0]
        == 4
    ),
    "all_race_ethnicity_categories_retained": bool(
        set(
            RACE_ETHNICITY_LEVELS
        )
        == set(
            participant_analysis[
                "race_ethnicity"
            ].astype(str).unique()
        )
    ),
    "fixed_income_groups_used": bool(
        set(
            INCOME_GROUP_LEVELS
        )
        == set(
            participant_analysis[
                "income_poverty_group"
            ].astype(str).unique()
        )
    ),
    "absolute_point_rows": int(
        len(
            absolute_point_estimates
        )
    ),
    "expected_absolute_point_rows": int(
        expected_absolute_rows
    ),
    "bootstrap_replicates_requested": (
        N_BOOTSTRAP
    ),
    "table4_long_rows": int(
        len(
            primary_threshold_rows
        )
    ),
    "figure4_saved": bool(
        figure4_png_path.exists()
        and figure4_pdf_path.exists()
    ),
    "all_expected_outputs_saved": bool(
        len(
            missing_output_files
        )
        == 0
    ),
    "metadata_saved": bool(
        FAIRNESS_METADATA_PATH.exists()
    ),
}

if (
    final_checkpoint[
        "absolute_point_rows"
    ]
    != final_checkpoint[
        "expected_absolute_point_rows"
    ]
):
    raise ValueError(
        "The number of absolute subgroup metric rows is unexpected."
    )

print("Saved Notebook 07 metadata to:")
print(FAIRNESS_METADATA_PATH)
print()
print("Final checkpoint:")
final_checkpoint


## Completion criteria

- Every gap is linked to a specific target, operating point, and reference group.
- Subgroup and target-positive counts reconcile with the analytic sample.
- All fairness tables, figures, and metadata are written.